# 05 — Final Load Preparation & KPI Computation

**Project**: Hospital Readmission Rate Analysis (Diabetes 130-US Hospitals)  
**Team**: SEC-A, Group 3  
**Sector**: Healthcare  

---

## Objective

This notebook is the **final step** in the ETL pipeline. It:

1. Computes **KPI metrics** aligned with the business problem
2. Creates **aggregated views** optimised for Tableau dashboarding
3. Exports **Tableau-ready CSV files** to `data/processed/`
4. Generates the **final data summary** for the project report

### KPI Framework

| KPI | Formula | Business Relevance |
|-----|---------|--------------------|
| 30-Day Readmission Rate | `readmitted_binary = 1` / total | Primary CMS penalty metric |
| Overall Readmission Rate | `readmitted ≠ NO` / total | Broad readmission burden |
| Avg Length of Stay (LOS) | mean(`time_in_hospital`) | Operational efficiency |
| Comorbidity Index | mean(`number_diagnoses`) | Patient complexity measure |
| Lab Testing Coverage | % with HbA1c or glucose test | Care quality indicator |
| High-Risk Patient Proportion | % `high_utiliser = 1` | Care management caseload |
| Medication Change Rate | % with `med_changed = 1` | Treatment adjustment frequency |

> **Input**: `data/processed/cleaned_healthcare.csv`  
> **Output**: Multiple Tableau-ready files in `data/processed/`

## 1. Environment Setup

In [1]:
# ============================================================
# 05_final_load_prep.ipynb — Environment Setup
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Final load preparation started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Final load preparation started at: 2026-04-21 13:53:08


In [2]:
# ============================================================
# Load cleaned dataset
# ============================================================

INPUT_PATH = os.path.join(PROCESSED_DIR, 'cleaned_healthcare.csv')
df = pd.read_csv(INPUT_PATH)

print(f"✅ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Data loaded: 71,515 rows × 50 columns


## 2. KPI Computation — Overall Metrics

In [3]:
# ============================================================
# 2a. Compute headline KPIs
# ============================================================

total_encounters = len(df)

kpis = {
    'Total Encounters': total_encounters,
    '30-Day Readmission Rate (%)': round(df['readmitted_binary'].mean() * 100, 2),
    'Overall Readmission Rate (%)': round((df['readmitted'] != 'NO').mean() * 100, 2),
    'Avg Length of Stay (days)': round(df['time_in_hospital'].mean(), 2),
    'Median Length of Stay (days)': round(df['time_in_hospital'].median(), 2),
    'Avg Number of Diagnoses': round(df['number_diagnoses'].mean(), 2),
    'Avg Number of Medications': round(df['num_medications'].mean(), 2),
    'HbA1c Testing Coverage (%)': round(df['a1c_test_performed'].mean() * 100, 2),
    'Glucose Testing Coverage (%)': round(df['glu_test_performed'].mean() * 100, 2),
    'High-Risk Patient Proportion (%)': round(df['high_utiliser'].mean() * 100, 2),
    'Medication Change Rate (%)': round(df['med_changed'].mean() * 100, 2),
    'Diabetes Medication Prescribed (%)': round(df['diabetes_med_flag'].mean() * 100, 2),
    'Primary Diagnosis = Diabetes (%)': round(df['primary_diag_diabetes'].mean() * 100, 2),
    'Avg Prior Inpatient Visits': round(df['number_inpatient'].mean(), 2),
    'Avg Prior Emergency Visits': round(df['number_emergency'].mean(), 2),
    'Avg Prior Outpatient Visits': round(df['number_outpatient'].mean(), 2),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])

print("\n" + "=" * 55)
print("         HEADLINE KPI DASHBOARD")
print("=" * 55)
for kpi, val in kpis.items():
    print(f"  {kpi:45s}: {val}")
print("=" * 55)


         HEADLINE KPI DASHBOARD
  Total Encounters                             : 71515
  30-Day Readmission Rate (%)                  : 8.8
  Overall Readmission Rate (%)                 : 39.9
  Avg Length of Stay (days)                    : 4.29
  Median Length of Stay (days)                 : 3.0
  Avg Number of Diagnoses                      : 7.25
  Avg Number of Medications                    : 15.7
  HbA1c Testing Coverage (%)                   : 18.16
  Glucose Testing Coverage (%)                 : 4.83
  High-Risk Patient Proportion (%)             : 3.79
  Medication Change Rate (%)                   : 44.78
  Diabetes Medication Prescribed (%)           : 75.95
  Primary Diagnosis = Diabetes (%)             : 8.12
  Avg Prior Inpatient Visits                   : 0.18
  Avg Prior Emergency Visits                   : 0.1
  Avg Prior Outpatient Visits                  : 0.28


## 3. Aggregated Views for Tableau

We create pre-aggregated tables that Tableau can load directly for dashboard construction.

In [4]:
# ============================================================
# 3a. View 1: Readmission by Demographics (Age × Gender × Race)
# ============================================================

demo_view = df.groupby(['age_group', 'gender', 'race']).agg(
    total_encounters=('readmitted_binary', 'count'),
    readmissions_30d=('readmitted_binary', 'sum'),
    readmit_rate_30d=('readmitted_binary', 'mean'),
    avg_length_of_stay=('time_in_hospital', 'mean'),
    avg_num_diagnoses=('number_diagnoses', 'mean'),
    avg_num_medications=('num_medications', 'mean'),
    avg_prior_inpatient=('number_inpatient', 'mean')
).reset_index()

demo_view['readmit_rate_30d'] = (demo_view['readmit_rate_30d'] * 100).round(2)
demo_view = demo_view.round(2)

print(f"Demographics view: {demo_view.shape[0]} rows")
demo_view.head(10)

Demographics view: 116 rows


,age_group,gender,race,total_encounters,readmissions_30d,readmit_rate_30d,avg_length_of_stay,avg_num_diagnoses,avg_num_medications,avg_prior_inpatient
0,[0-10),Female,AfricanAmerican,8,0,0.0000,2.6200,2.6200,5.6200,0.0000
1,[0-10),Female,Asian,1,0,0.0000,4.0000,3.0000,9.0000,0.0000
2,[0-10),Female,Caucasian,69,1,1.4500,2.7100,3.0100,7.1300,0.0900
3,[0-10),Female,Hispanic,2,0,0.0000,2.5000,3.5000,4.0000,0.0000
4,[0-10),Female,Unknown,1,0,0.0000,2.0000,1.0000,4.0000,0.0000
5,[0-10),Male,AfricanAmerican,8,0,0.0000,3.0000,3.2500,7.1200,0.0000
6,[0-10),Male,Asian,1,0,0.0000,1.0000,2.0000,7.0000,0.0000
7,[0-10),Male,Caucasian,60,2,3.3300,2.4000,2.3500,5.3500,0.0200
8,[0-10),Male,Other,4,0,0.0000,3.0000,2.7500,6.5000,0.0000
9,[10-20),Female,AfricanAmerican,89,3,3.3700,2.9400,3.8500,7.3400,0.0800


In [5]:
# ============================================================
# 3b. View 2: Readmission by Clinical Factors
# ============================================================

clinical_view = df.groupby(['admission_type', 'discharge_disposition', 'diag_1_category']).agg(
    total_encounters=('readmitted_binary', 'count'),
    readmissions_30d=('readmitted_binary', 'sum'),
    readmit_rate_30d=('readmitted_binary', 'mean'),
    avg_length_of_stay=('time_in_hospital', 'mean'),
    avg_num_procedures=('num_procedures', 'mean'),
    avg_lab_procedures=('num_lab_procedures', 'mean')
).reset_index()

clinical_view['readmit_rate_30d'] = (clinical_view['readmit_rate_30d'] * 100).round(2)
clinical_view = clinical_view.round(2)

print(f"Clinical view: {clinical_view.shape[0]} rows")
clinical_view.head(10)

Clinical view: 896 rows


,admission_type,discharge_disposition,diag_1_category,total_encounters,readmissions_30d,readmit_rate_30d,avg_length_of_stay,avg_num_procedures,avg_lab_procedures
0,Elective,AMA (Against Medical Advice),Circulatory System,11,2,18.1800,3.4500,2.3600,41.7300
1,Elective,AMA (Against Medical Advice),Digestive System,1,0,0.0000,1.0000,2.0000,47.0000
2,Elective,AMA (Against Medical Advice),Endocrine/Metabolic (incl. Diabetes),3,1,33.3300,1.3300,0.0000,31.3300
3,Elective,AMA (Against Medical Advice),Genitourinary System,1,0,0.0000,1.0000,0.0000,34.0000
4,Elective,AMA (Against Medical Advice),Injury & Poisoning,1,0,0.0000,2.0000,6.0000,42.0000
5,Elective,AMA (Against Medical Advice),Mental Disorders,3,0,0.0000,6.0000,1.3300,40.0000
6,Elective,AMA (Against Medical Advice),Musculoskeletal,1,0,0.0000,8.0000,1.0000,55.0000
7,Elective,AMA (Against Medical Advice),Pregnancy & Childbirth,2,0,0.0000,4.0000,1.0000,63.5000
8,Elective,AMA (Against Medical Advice),Respiratory System,1,0,0.0000,6.0000,1.0000,1.0000
9,Elective,AMA (Against Medical Advice),Skin & Subcutaneous,1,0,0.0000,2.0000,1.0000,49.0000


In [6]:
# ============================================================
# 3c. View 3: Medication & Treatment Patterns
# ============================================================

med_view = df.groupby(['insulin', 'med_changed', 'diabetes_med_flag', 'a1c_test_performed']).agg(
    total_encounters=('readmitted_binary', 'count'),
    readmissions_30d=('readmitted_binary', 'sum'),
    readmit_rate_30d=('readmitted_binary', 'mean'),
    avg_active_meds=('n_active_medications', 'mean'),
    avg_med_changes=('n_med_changes', 'mean')
).reset_index()

med_view['readmit_rate_30d'] = (med_view['readmit_rate_30d'] * 100).round(2)
med_view = med_view.round(2)

print(f"Medication view: {med_view.shape[0]} rows")
med_view.head(10)

Medication view: 14 rows


,insulin,med_changed,diabetes_med_flag,a1c_test_performed,total_encounters,readmissions_30d,readmit_rate_30d,avg_active_meds,avg_med_changes
0,Down,1,1,0,5618,603,10.7300,1.5000,1.0400
1,Down,1,1,1,1887,172,9.1100,1.6300,1.1000
2,No,0,0,0,14968,1097,7.3300,0.0000,0.0000
3,No,0,0,1,2230,166,7.4400,0.0000,0.0000
4,No,0,1,0,9462,865,9.1400,0.9700,0.0000
5,No,0,1,1,1517,150,9.8900,0.9700,0.0000
6,No,1,1,0,5565,469,8.4300,1.9500,0.2600
7,No,1,1,1,1176,95,8.0800,1.9400,0.3700
8,Steady,0,1,0,9225,886,9.6000,1.0000,0.0000
9,Steady,0,1,1,2091,150,7.1700,1.0000,0.0000


In [7]:
# ============================================================
# 3d. View 4: Specialty Performance
# ============================================================

specialty_view = df.groupby('medical_specialty_grouped').agg(
    total_encounters=('readmitted_binary', 'count'),
    readmissions_30d=('readmitted_binary', 'sum'),
    readmit_rate_30d=('readmitted_binary', 'mean'),
    avg_length_of_stay=('time_in_hospital', 'mean'),
    avg_num_diagnoses=('number_diagnoses', 'mean'),
    avg_num_medications=('num_medications', 'mean'),
    pct_diabetes_primary=('primary_diag_diabetes', 'mean')
).sort_values('total_encounters', ascending=False).reset_index()

specialty_view['readmit_rate_30d'] = (specialty_view['readmit_rate_30d'] * 100).round(2)
specialty_view['pct_diabetes_primary'] = (specialty_view['pct_diabetes_primary'] * 100).round(2)
specialty_view = specialty_view.round(2)

print(f"Specialty view: {specialty_view.shape[0]} rows")
specialty_view

Specialty view: 10 rows


,medical_specialty_grouped,total_encounters,readmissions_30d,readmit_rate_30d,avg_length_of_stay,avg_num_diagnoses,avg_num_medications,pct_diabetes_primary
0,Other,40691,3606,8.8600,4.3500,7.4200,16.0400,7.9900
1,InternalMedicine,10919,1041,9.5300,4.5200,6.9700,14.3700,11.1500
2,Family/GeneralPractice,5118,488,9.5300,4.3000,6.9800,13.8200,11.4300
3,Emergency/Trauma,4465,344,7.7000,4.2100,7.6000,13.5600,8.3100
4,Cardiology,4265,303,7.1000,3.4900,7.0000,16.8500,1.3100
5,Surgery-General,2221,184,8.2800,4.3500,6.6200,16.4600,6.2600
6,Orthopedics,1134,112,9.8800,3.8700,6.3500,20.6200,0.9700
7,Orthopedics-Reconstructive,1043,69,6.6200,3.8100,6.2600,20.2000,2.4000
8,Radiologist,831,58,6.9800,3.4000,7.6300,18.6200,1.5600
9,Nephrology,828,88,10.6300,4.8300,7.1900,16.8500,16.4300


In [8]:
# ============================================================
# 3e. View 5: Risk Segmentation (from clustering in NB04)
# ============================================================

# Re-compute segments (since NB04 is separate)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

cluster_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_inpatient', 'number_emergency',
    'number_diagnoses', 'age_midpoint', 'total_prior_visits',
    'n_active_medications'
]

X_cluster = df[cluster_features].fillna(0)
X_scaled = StandardScaler().fit_transform(X_cluster)
df['risk_segment'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)

# Label segments by readmission risk
segment_risk = df.groupby('risk_segment')['readmitted_binary'].mean().sort_values()
risk_labels = {seg: label for seg, label in zip(
    segment_risk.index,
    ['Low Risk', 'Moderate Risk', 'Elevated Risk', 'High Risk']
)}
df['risk_label'] = df['risk_segment'].map(risk_labels)

segment_view = df.groupby('risk_label').agg(
    total_encounters=('readmitted_binary', 'count'),
    readmissions_30d=('readmitted_binary', 'sum'),
    readmit_rate_30d=('readmitted_binary', 'mean'),
    avg_age=('age_midpoint', 'mean'),
    avg_length_of_stay=('time_in_hospital', 'mean'),
    avg_num_diagnoses=('number_diagnoses', 'mean'),
    avg_prior_inpatient=('number_inpatient', 'mean'),
    avg_medications=('num_medications', 'mean'),
    pct_high_utiliser=('high_utiliser', 'mean')
).reset_index()

segment_view['readmit_rate_30d'] = (segment_view['readmit_rate_30d'] * 100).round(2)
segment_view['pct_high_utiliser'] = (segment_view['pct_high_utiliser'] * 100).round(2)
segment_view = segment_view.round(2)

print(f"Risk segment view: {segment_view.shape[0]} rows")
segment_view

Risk segment view: 4 rows


,risk_label,total_encounters,readmissions_30d,readmit_rate_30d,avg_age,avg_length_of_stay,avg_num_diagnoses,avg_prior_inpatient,avg_medications,pct_high_utiliser
0,Elevated Risk,14838,1529,10.3000,66.5500,7.8100,8.1400,0.1000,25.6100,0.9200
1,High Risk,4598,754,16.4000,65.7400,4.5300,7.9300,1.6700,16.2700,45.3700
2,Low Risk,21404,1350,6.3100,56.0800,2.7900,4.9500,0.0500,11.6200,0.6800
3,Moderate Risk,30675,2660,8.6700,71.8900,3.5900,8.3200,0.0800,13.6800,1.1200


## 4. Export Tableau-Ready Files

In [9]:
# ============================================================
# 4. Save all views as CSV for Tableau
# ============================================================

exports = {
    'tableau_master.csv': df,
    'tableau_demographics.csv': demo_view,
    'tableau_clinical.csv': clinical_view,
    'tableau_medication.csv': med_view,
    'tableau_specialty.csv': specialty_view,
    'tableau_risk_segments.csv': segment_view,
    'kpi_summary.csv': kpi_df
}

print("=" * 60)
print("         EXPORTED FILES")
print("=" * 60)

for filename, data in exports.items():
    filepath = os.path.join(PROCESSED_DIR, filename)
    data.to_csv(filepath, index=False)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  ✅ {filename:35s} | {data.shape[0]:>7,} rows × {data.shape[1]:>3} cols | {size_kb:.1f} KB")

print("=" * 60)
print(f"\nAll files saved to: {os.path.abspath(PROCESSED_DIR)}")

         EXPORTED FILES


  ✅ tableau_master.csv                  |  71,515 rows ×  52 cols | 22823.3 KB
  ✅ tableau_demographics.csv            |     116 rows ×  10 cols | 6.3 KB
  ✅ tableau_clinical.csv                |     896 rows ×   9 cols | 66.6 KB
  ✅ tableau_medication.csv              |      14 rows ×   9 cols | 0.6 KB
  ✅ tableau_specialty.csv               |      10 rows ×   8 cols | 0.6 KB
  ✅ tableau_risk_segments.csv           |       4 rows ×  10 cols | 0.4 KB
  ✅ kpi_summary.csv                     |      16 rows ×   2 cols | 0.5 KB

All files saved to: /Users/pankajbaid/pankajbaid567/SEC-A_g-3_Hospital_readmission_Rate/data/processed


## 5. Final Column Reference

Complete reference of all columns in the master Tableau file.

In [10]:
# ============================================================
# 5. Column reference for the master file
# ============================================================

col_ref = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.values,
    'non_null': df.count().values,
    'null_count': df.isnull().sum().values,
    'unique_count': df.nunique().values,
    'sample': [df[col].iloc[0] for col in df.columns]
})

col_ref

,column,dtype,non_null,null_count,unique_count,sample
0,encounter_id,int64,71515,0,71515,12522
1,patient_nbr,int64,71515,0,71515,48330783
2,race,object,71515,0,6,Caucasian
3,gender,object,71515,0,2,Female
4,time_in_hospital,int64,71515,0,14,13
5,medical_specialty,object,71515,0,71,Other
6,num_lab_procedures,int64,71515,0,116,68
7,num_procedures,int64,71515,0,7,2
8,num_medications,int64,71515,0,75,28
9,number_outpatient,int64,71515,0,33,0


## 6. Tableau Dashboard Design Recommendations

Based on the KPIs and aggregated views, the Tableau dashboard should contain:

### Executive Summary View
- **KPI Tiles**: 30-day readmission rate, total encounters, avg LOS, HbA1c coverage
- **Readmission Trend**: Bar chart of readmission status distribution
- **Risk Heatmap**: Age group × admission type coloured by readmission rate

### Operational Drill-Down View
- **Demographic Filters**: Age, gender, race (interactive)
- **Clinical Filters**: Admission type, discharge disposition, primary diagnosis
- **Specialty Comparison**: Horizontal bar by medical specialty
- **Medication Impact**: Insulin status × medication change vs readmission rate

### Risk Segmentation View
- **Segment Distribution**: Pie chart of patient risk segments
- **Segment Profile Cards**: Key metrics per segment
- **Actionable Callouts**: Highlight high-risk segment characteristics

### Interactive Elements Required
- ☐ Age group filter
- ☐ Gender filter
- ☐ Admission type filter
- ☐ Diagnosis category filter
- ☐ Risk segment selector

## 7. Business Recommendations Summary

Based on the complete ETL → EDA → Statistical Analysis pipeline:

### Recommendation 1: Implement a Readmission Risk Score at Admission
- **Insight**: Prior inpatient visits, number of diagnoses, and age are the strongest predictors
- **Action**: Deploy a simple scoring model using these 3 variables to flag high-risk patients at admission
- **Impact**: Early identification enables proactive discharge planning, estimated to reduce 30-day readmissions by 10–15%

### Recommendation 2: Mandate HbA1c Testing for All Diabetic Admissions
- **Insight**: <17% of diabetic patients receive HbA1c testing during their stay
- **Action**: Implement a clinical protocol requiring HbA1c measurement for every diabetic admission
- **Impact**: Improved glycaemic management during hospitalisation and better-informed discharge medication decisions

### Recommendation 3: Target High Utilisers with Transitional Care Programmes
- **Insight**: High utilisers (>3 prior visits) have significantly elevated readmission rates
- **Action**: Assign care coordinators to high-utiliser patients for 30-day post-discharge follow-up
- **Impact**: Reduce repeat admissions in the costliest patient cohort; estimated $2,000–$5,000 savings per prevented readmission

### Recommendation 4: Strengthen Home Discharge Protocols
- **Insight**: Patients discharged to home have higher readmission rates than those discharged to SNF/rehab
- **Action**: Implement structured phone follow-up within 48 hours for all home-discharged patients
- **Impact**: Catch deterioration early before it leads to ER visits and readmission

### Recommendation 5: Address Racial Disparities in Post-Discharge Care
- **Insight**: African American patients show elevated readmission rates, suggesting disparities in follow-up care access
- **Action**: Partner with community health organisations to provide culturally competent post-discharge support
- **Impact**: Reduce health equity gaps while lowering readmission rates in underserved populations

In [11]:
# ============================================================
# 7. Final pipeline status
# ============================================================

print("\n" + "=" * 60)
print("       ✅ ETL PIPELINE COMPLETE")
print("=" * 60)
print(f"  Raw data:      data/raw/RAW_HEALTHCARE.csv")
print(f"  Cleaned data:  data/processed/cleaned_healthcare.csv")
print(f"  Tableau files: data/processed/tableau_*.csv")
print(f"  KPI summary:   data/processed/kpi_summary.csv")
print(f"")
print(f"  Notebooks executed:")
print(f"    01_extraction.ipynb        ✅")
print(f"    02_cleaning.ipynb           ✅")
print(f"    03_eda.ipynb                ✅")
print(f"    04_statistical_analysis.ipynb ✅")
print(f"    05_final_load_prep.ipynb     ✅")
print(f"")
print(f"  Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


       ✅ ETL PIPELINE COMPLETE
  Raw data:      data/raw/RAW_HEALTHCARE.csv
  Cleaned data:  data/processed/cleaned_healthcare.csv
  Tableau files: data/processed/tableau_*.csv
  KPI summary:   data/processed/kpi_summary.csv

  Notebooks executed:
    01_extraction.ipynb        ✅
    02_cleaning.ipynb           ✅
    03_eda.ipynb                ✅
    04_statistical_analysis.ipynb ✅
    05_final_load_prep.ipynb     ✅

  Completed at: 2026-04-21 13:53:10
